In [1]:
%load_ext line_profiler

In [1]:
from qiskit.quantum_info import SparsePauliOp, PauliList
from qiskit._accelerate.sparse_pauli_op import unordered_unique
import numpy as np

In [2]:
def compose_IZ_only(op_1: SparsePauliOp, op_2: SparsePauliOp) -> SparsePauliOp:
    """
    Compose two SparsePauliOps containing only I and Z terms.
    
    Args:
        op_1: SparsePauliOp containing only I and Z terms.
        op_2: SparsePauliOp containing only I and Z terms.
    
    Returns:
        SparsePauliOp representing the composition.
    """
    # assert op_1.num_qubits == op_2.num_qubits, "Mismatched qubit dimensions"
    num_qubits = op_1.num_qubits

    # Combine Z terms: XOR for Z locations, AND for phase calculation (not needed)
    z1, z2 = op_1.paulis.z, op_2.paulis.z

    # XOR to determine Z locations in the product
    z_combined = np.logical_xor(z1[:, np.newaxis], z2).reshape((-1, num_qubits))
    
    # Create new Pauli list with combined Z locations and phases
    pauli_list = PauliList.from_symplectic(z_combined, np.zeros_like(z_combined))
    # Combine coefficients
    coeffs = np.multiply.outer(op_1.coeffs, op_2.coeffs).ravel()

    return SparsePauliOp(pauli_list, coeffs, copy=False)


def compose_IZ_only_optimized(op_1: SparsePauliOp, op_2: SparsePauliOp) -> SparsePauliOp:
    """
    Compose two SparsePauliOps containing only I and Z terms.
    
    Args:
        op_1: SparsePauliOp containing only I and Z terms.
        op_2: SparsePauliOp containing only I and Z terms.
    
    Returns:
        SparsePauliOp representing the composition.
    """
    # assert op_1.num_qubits == op_2.num_qubits, "Mismatched qubit dimensions"
    num_qubits = op_1.num_qubits

    # Combine Z terms: XOR for Z locations, AND for phase calculation (not needed)
    z1, z2 = op_1.paulis.z, op_2.paulis.z

    # XOR to determine Z locations in the product
    z_combined = np.logical_xor(z1[:, np.newaxis], z2).reshape((-1, num_qubits))
    # Multiply out coefficients
    coeffs = np.multiply.outer(op_1.coeffs, op_2.coeffs).ravel()
    # # Filter non-zero coefficients
    # non_zero = np.logical_not(np.isclose(coeffs, 0, atol=1e-8, rtol=1e-5))
    # z_combined = z_combined[non_zero]
    # nz_coeffs = coeffs[non_zero]

    # Simplify the pauli op here
    array = np.packbits(z_combined, axis=1).astype(np.uint16)
    # Find unique Pauli terms
    indices, inverses = unordered_unique(array)
    # if np.all(non_zero) and indices.shape[0] == array.shape[0]:
    #     # No zero operator or duplicate operator
    #     pauli_list = PauliList.from_symplectic(z_combined, np.zeros_like(z_combined))
    #     return SparsePauliOp(pauli_list, nz_coeffs, copy=False)

    if indices.shape[0] == array.shape[0]:
        # No duplicate operator
        pauli_list = PauliList.from_symplectic(z_combined, np.zeros_like(z_combined))
        return SparsePauliOp(pauli_list, coeffs, copy=False)
    
    coeffs_combined = np.zeros(indices.shape[0], dtype=coeffs.dtype)
    # Combine coefficients of duplicated Pauli terms
    np.add.at(coeffs_combined, inverses, coeffs)
    is_zero = np.isclose(coeffs_combined, 0, atol=1e-8, rtol=1e-5)
    # Check the edge case that we deleted all Paulis
    # In this case we return an identity Pauli with a zero coefficient
    if np.all(is_zero):
        z = np.zeros((1, num_qubits), dtype=bool)
        coeffs = np.array([0j], dtype=coeffs.dtype)
    else:
        non_zero = np.logical_not(is_zero)
        nz_indices = indices[non_zero]
        z = z_combined[nz_indices]
        coeffs = coeffs_combined[non_zero]
    
    # Create new Pauli list with combined Z locations and phases
    pauli_list = PauliList.from_symplectic(z, np.zeros_like(z))

    return SparsePauliOp(pauli_list, coeffs, ignore_pauli_phase=True, copy=False)

In [64]:
op1 = SparsePauliOp.from_list([('ZIIZI', 0.2), ('IZZZI', 0.8), ('IZIZI', 0.1)])
op2 = SparsePauliOp.from_list([('IZZZI', 0.4), ('ZIIZI', 0.5), ('IZIZI', 0.6)])

z1, z2 = op1.paulis.z, op2.paulis.z
# print(z1)
# print(z2)
num_qubits = op1.num_qubits
z_combined = np.logical_xor(z1[:, np.newaxis], z2).reshape((-1, num_qubits))
print(z_combined)
coeffs = np.multiply.outer(op1.coeffs, op2.coeffs).ravel()
print(coeffs)

[[False False  True  True  True]
 [False False False False False]
 [False False False  True  True]
 [False False False False False]
 [False False  True  True  True]
 [False False  True False False]
 [False False  True False False]
 [False False False  True  True]
 [False False False False False]]
[0.08+0.j 0.1 +0.j 0.12+0.j 0.32+0.j 0.4 +0.j 0.48+0.j 0.04+0.j 0.05+0.j
 0.06+0.j]


In [22]:
non_zero = np.logical_not(np.isclose(coeffs, 0, atol=1e-8, rtol=1e-5))
print(non_zero)

[ True  True  True  True  True  True  True  True  True]


In [46]:
# SparsePauliOp for 20 qubits
dist_op = SparsePauliOp.from_list([('IIIIZIIIIIIIIIIIIIII', 1.0), ('IZIZIIIIIIIIIIIIIIII', 0.5), ('IIIIIIIIIIIIIIIIIIIZ', 1.2), ('IIIIIIIIIIIIIIIIIZII', 0.3), ('IIIIIIIIIIIIIIIZIIII', 0.8), ('IIIIIIIIIIIIIZIIIIII', 1.1), ('IIIIIIIIIZIIIIIIIIII', 0.4), ('IIIIIZIIIIIIIIIIIIII', 0.1), ('ZIIIIIIIIIIIIIIIIIII', 0.6), ('IIIIIIIIIIIIIZIIIIII', 0.9), ('IIIIZIZIIIIIIZIIIIII', 0.7), ('IIIIIIIZIZIIIZIIIIII', 0.2), ('IIIZIIIIIZZIIZIIIIII', 1.2)])
h_olap_1 = SparsePauliOp.from_list([('IIIIIIIIIIIIIIIIIIII', 1.0)])

for k in range(1, 100):
    h_op = dist_op
    for _ in range(k - 1):
        h_op = (h_op @ dist_op).simplify()
    h_olap_1 = SparsePauliOp.sum([h_olap_1, h_op]).simplify()

In [47]:
len(h_olap_1)

4096

In [48]:
h_olap_2 = SparsePauliOp.from_list([('IIIIIIIIIIIIIIIIIIII', 1.0)])

for k in range(1, 100):
    if k == 1:
        h_op = dist_op
    else:
        h_op = compose_IZ_only_optimized(h_op, dist_op)
    h_olap_2 = SparsePauliOp.sum([h_olap_2, h_op]).simplify()

In [49]:
len(h_olap_2)

4096

In [50]:
(h_olap_1.simplify()).equiv(h_olap_2.simplify())

True

In [45]:
# Profile the function above
%lprun -u 1e-1 -f compose_IZ_only_optimized compose_IZ_only_optimized(h_olap_1, h_olap_2)

Timer unit: 0.1 s

Total time: 4.87105 s
File: /var/folders/f0/3f43g55s7jj8xnhyp1dq2nfh0000gn/T/ipykernel_24147/1267588333.py
Function: compose_IZ_only_optimized at line 29

Line #      Hits         Time  Per Hit   % Time  Line Contents
    29                                           def compose_IZ_only_optimized(op_1: SparsePauliOp, op_2: SparsePauliOp) -> SparsePauliOp:
    30                                               """
    31                                               Compose two SparsePauliOps containing only I and Z terms.
    32                                               
    33                                               Args:
    34                                                   op_1: SparsePauliOp containing only I and Z terms.
    35                                                   op_2: SparsePauliOp containing only I and Z terms.
    36                                               
    37                                               Returns:
    38     

TEST_1 Total time: 6.64017 s